# Coupling simulator parameter sliders

Generate distinct participant histories across trajectory rounds, hold each participant's final trial fixed across those rounds, adjust the simulator parameters with sliders, and redraw the source arrows with the final-trial simulated choice distribution from `simulate_coupling`.

In [ ]:
import os
from pathlib import Path
import sys

MPLCONFIGDIR = Path("/tmp/coupling_sources_matplotlib")
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from ipywidgets import BoundedIntText, FloatSlider, HBox, VBox, interactive_output
from matplotlib.lines import Line2D

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "coupling_sources").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from coupling_sources.core.data import StimulusBatch, StimulusMetadata, StimulusSpec
from coupling_sources.core.simulator import simulate_coupling
from coupling_sources.core.stimuli import generate_stimulus
from coupling_sources.utils.angles import optimal_weights

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.facecolor"] = "white"

In [ ]:
N_GENERATED_PARTICIPANTS = 30
N_TRAJECTORY_ROUNDS = 500
RESPONSE_SEED = 201


def repeat_final_trial_across_rounds(batch):
    final_trial = -1
    return StimulusBatch(
        visible_reports=batch.visible_reports.at[:, :, final_trial, :].set(batch.visible_reports[:, :1, final_trial, :]),
        hidden_source=batch.hidden_source.at[:, :, final_trial].set(batch.hidden_source[:, :1, final_trial]),
        trial_type=batch.trial_type.at[:, :, final_trial].set(batch.trial_type[:, :1, final_trial]),
        valid_trials=batch.valid_trials.at[:, :, final_trial].set(batch.valid_trials[:, :1, final_trial]),
        metadata=StimulusMetadata(
            complete_reports=batch.metadata.complete_reports.at[:, :, final_trial, :].set(batch.metadata.complete_reports[:, :1, final_trial, :]),
            true_direction=batch.metadata.true_direction.at[:, :, final_trial].set(batch.metadata.true_direction[:, :1, final_trial]),
            source_choices=batch.metadata.source_choices.at[:, :, final_trial, :].set(batch.metadata.source_choices[:, :1, final_trial, :]),
            stimulus_covariance=batch.metadata.stimulus_covariance,
            resultant_length=batch.metadata.resultant_length.at[:, :, final_trial].set(batch.metadata.resultant_length[:, :1, final_trial]),
        ),
    )


STIMULUS_SPEC = StimulusSpec(
    seed=101,
    n_participants=N_GENERATED_PARTICIPANTS,
    n_sources=4,
    n_rounds=N_TRAJECTORY_ROUNDS,
    n_trials_full=20,
    n_trials_partial=20,
    sigma=0.55,
    lam=0.8,
    resultant_floor=0.6,
    coupled_sources=(0, 1),
)

base_stimuli = generate_stimulus(STIMULUS_SPEC)
stimuli = repeat_final_trial_across_rounds(base_stimuli)
N_PARTICIPANTS, N_ROUNDS, N_TRIALS, N_SOURCES = stimuli.visible_reports.shape
FINAL_TRIAL_INDEX = N_TRIALS - 1
SOURCE_COLORS = ["#2563eb", "#16a34a", "#f97316", "#7c3aed", "#0891b2", "#be123c"]
SOURCE_LABELS = [f"S{source}" for source in range(N_SOURCES)]
OPTIMAL_SOURCE_WEIGHTS = np.asarray(optimal_weights(stimuli.metadata.stimulus_covariance))


def mean_relative_agreement_matrix(visible_reports):
    visible_reports = np.asarray(visible_reports, dtype=float)
    n_sources = visible_reports.shape[-1]
    off_diagonal = ~np.eye(n_sources, dtype=bool)
    visible_sources = np.isfinite(visible_reports)
    reports = np.nan_to_num(visible_reports, nan=0.0)
    visible_pairs = visible_sources[..., :, None] & visible_sources[..., None, :] & off_diagonal
    pair_agreement = np.cos(reports[..., :, None] - reports[..., None, :])
    pair_count = np.maximum(np.sum(visible_pairs, axis=(-2, -1), keepdims=True), 1)
    mean_visible_agreement = np.sum(pair_agreement * visible_pairs, axis=(-2, -1), keepdims=True) / pair_count
    relative_agreement = 0.5 * (pair_agreement - mean_visible_agreement)
    sums = np.sum(np.where(visible_pairs, relative_agreement, 0.0), axis=(0, 1))
    counts = np.sum(visible_pairs, axis=(0, 1))
    means = np.divide(sums, counts, out=np.full((n_sources, n_sources), np.nan), where=counts > 0)
    np.fill_diagonal(means, np.nan)
    return means


REALIZED_STIMULUS_RELATIVE_AGREEMENTS = np.stack([
    mean_relative_agreement_matrix(base_stimuli.visible_reports[participant])
    for participant in range(N_PARTICIPANTS)
])

print(
    "stimuli:",
    f"participants={N_PARTICIPANTS}",
    f"trajectory_rounds={N_ROUNDS}",
    f"trials={N_TRIALS}",
    f"sources={N_SOURCES}",
)

In [ ]:
def angle_xy(angle, radius=1.0):
    return radius * np.cos(angle), radius * np.sin(angle)


def circular_mean_angle(angles, weights=None):
    angles = np.asarray(angles, dtype=float)
    finite = np.isfinite(angles)
    if not finite.any():
        return np.nan
    clean_angles = np.where(finite, angles, 0.0)
    if weights is None:
        effective_weights = finite.astype(float)
    else:
        effective_weights = np.broadcast_to(np.asarray(weights, dtype=float), angles.shape)
        effective_weights = np.where(finite, effective_weights, 0.0)
    x = np.sum(effective_weights * np.cos(clean_angles))
    y = np.sum(effective_weights * np.sin(clean_angles))
    if np.isclose(x, 0.0) and np.isclose(y, 0.0):
        return np.nan
    return np.arctan2(y, x) % (2.0 * np.pi)


def draw_angle_arrow(
    ax,
    angle,
    *,
    color,
    radius=0.92,
    linewidth=2.6,
    linestyle="-",
    alpha=1.0,
    zorder=4,
):
    if not np.isfinite(angle):
        return
    x, y = angle_xy(angle, radius)
    ax.annotate(
        "",
        xy=(x, y),
        xytext=(0.0, 0.0),
        arrowprops={
            "arrowstyle": "-|>",
            "color": color,
            "lw": linewidth,
            "linestyle": linestyle,
            "mutation_scale": 16,
            "shrinkA": 0,
            "shrinkB": 0,
            "alpha": alpha,
        },
        zorder=zorder,
    )


def draw_circle_frame(ax):
    theta = np.linspace(0.0, 2.0 * np.pi, 361)
    ax.plot(np.cos(theta), np.sin(theta), color="#d1d5db", linewidth=1.4)
    ax.axhline(0.0, color="#e5e7eb", linewidth=0.8, zorder=0)
    ax.axvline(0.0, color="#e5e7eb", linewidth=0.8, zorder=0)
    ax.scatter([0.0], [0.0], s=20, color="#111827", zorder=5)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.28, 1.28)
    ax.set_ylim(-1.28, 1.28)
    ax.axis("off")


def visible_label(source, is_hidden):
    if is_hidden:
        return f"{SOURCE_LABELS[source]} hidden"
    return SOURCE_LABELS[source]


def draw_matrix_heatmap(ax, matrix, title, *, limit):
    matrix = np.asarray(matrix, dtype=float)
    cmap = plt.get_cmap("coolwarm").copy()
    cmap.set_bad("#f3f4f6")
    image = ax.imshow(np.ma.masked_invalid(matrix), cmap=cmap, vmin=-limit, vmax=limit)
    ax.set_xticks(np.arange(N_SOURCES), SOURCE_LABELS)
    ax.set_yticks(np.arange(N_SOURCES), SOURCE_LABELS)
    ax.set_title(title)
    ax.tick_params(length=0)
    for row in range(N_SOURCES):
        for column in range(N_SOURCES):
            value = matrix[row, column]
            if np.isfinite(value):
                text_color = "white" if abs(value) > 0.68 * limit else "#111827"
                label = f"{value:.2f}"
            else:
                text_color = "#6b7280"
                label = "--"
            ax.text(column, row, label, ha="center", va="center", color=text_color, fontsize=9)
    return image


def draw_simulation(learningRate, g_y, g_z, vmConcentration, c0, participant):
    participant = int(participant)
    history_round = 0
    trial_index = FINAL_TRIAL_INDEX
    parameters = {
        "learningRate": float(learningRate),
        "g_y": float(g_y),
        "g_z": float(g_z),
        "vmConcentration": float(vmConcentration),
        "c0": float(c0),
    }
    simulation = simulate_coupling(
        seed=RESPONSE_SEED,
        batch=stimuli,
        parameters=parameters,
    )

    visible_reports = np.asarray(stimuli.visible_reports[participant, history_round, trial_index])
    complete_reports = np.asarray(stimuli.metadata.complete_reports[participant, history_round, trial_index])
    naive_source_mean = circular_mean_angle(visible_reports)
    optimal_source_mean = circular_mean_angle(complete_reports, OPTIMAL_SOURCE_WEIGHTS)
    hidden_source = int(np.asarray(stimuli.hidden_source[participant, history_round, trial_index]))
    is_probe = hidden_source >= 0

    diagnostics = simulation.diagnostics
    responses = np.asarray(simulation.responses[participant, :, trial_index])
    weights_name = "probe_weights" if is_probe else "truth_weights"
    weights_by_round = np.asarray(getattr(diagnostics, weights_name)[participant, :, trial_index])
    weights = np.nanmean(weights_by_round, axis=0)
    learned_pair_state = np.nanmean(np.asarray(diagnostics.pair_scores[participant, :, trial_index]), axis=0)
    learned_pair_state = learned_pair_state.copy()
    np.fill_diagonal(learned_pair_state, np.nan)
    stimulus_relative_agreement = REALIZED_STIMULUS_RELATIVE_AGREEMENTS[participant]
    matrix_limit = min(1.0, max(0.2, float(np.nanmax(np.abs(np.stack([learned_pair_state, stimulus_relative_agreement]))))))
    choice_mean = circular_mean_angle(responses)

    fig = plt.figure(figsize=(13, 9.2), constrained_layout=True)
    grid = fig.add_gridspec(
        2,
        2,
        height_ratios=[1.45, 0.95],
        width_ratios=[1.25, 0.9],
    )
    ax_circle = fig.add_subplot(grid[0, 0])
    ax_weights = fig.add_subplot(grid[0, 1])
    matrix_grid = grid[1, :].subgridspec(1, 2, wspace=0.12)
    ax_learned = fig.add_subplot(matrix_grid[0, 0])
    ax_realized = fig.add_subplot(matrix_grid[0, 1])
    draw_circle_frame(ax_circle)

    finite_responses = np.isfinite(responses)
    if finite_responses.any():
        response_samples = responses[finite_responses]
        jitter = (np.arange(response_samples.size) * 0.61803398875) % 1.0
        sample_radius = 1.01 + 0.20 * jitter
        xs = sample_radius * np.cos(response_samples)
        ys = sample_radius * np.sin(response_samples)
        ax_circle.scatter(
            xs,
            ys,
            s=26,
            color="#ef4444",
            alpha=0.28,
            linewidths=0,
            zorder=2,
        )

    draw_angle_arrow(ax_circle, naive_source_mean, color="#374151", radius=1.0, linewidth=2.0, zorder=3)
    draw_angle_arrow(ax_circle, optimal_source_mean, color="#0f172a", radius=1.08, linewidth=2.2, linestyle="-.", alpha=0.86, zorder=8)

    for source, angle in enumerate(complete_reports):
        source_color = SOURCE_COLORS[source % len(SOURCE_COLORS)]
        source_hidden = source == hidden_source
        source_visible = np.isfinite(visible_reports[source])
        draw_angle_arrow(
            ax_circle,
            angle,
            color=source_color,
            radius=0.78,
            linewidth=3.0 if source_visible else 2.0,
            linestyle="--" if source_hidden else "-",
            alpha=1.0 if source_visible else 0.42,
            zorder=6 if source_visible else 2,
        )
        label_radius = 0.88 if source_visible else 0.68
        label_x, label_y = angle_xy(angle, radius=label_radius)
        ax_circle.text(
            label_x,
            label_y,
            visible_label(source, source_hidden and not source_visible),
            color=source_color,
            fontsize=10,
            ha="center",
            va="center",
            weight="bold" if source_visible else "normal",
            alpha=1.0 if source_visible else 0.65,
        )

    draw_angle_arrow(ax_circle, choice_mean, color="#dc2626", radius=1.0, linewidth=2.4, linestyle="-", alpha=0.8, zorder=9)

    trial_kind = "hidden-source probe" if is_probe else "truth estimate"
    choice_mean_label = "mean hidden-source prediction" if is_probe else "mean truth estimate"
    ax_circle.set_title(
        f"Participant {participant}, final trial {trial_index}: {trial_kind}",
        pad=12,
    )

    legend_items = [
        Line2D([0], [0], color="#374151", lw=2, label="naive visible-source mean"),
        Line2D([0], [0], color="#0f172a", lw=2.2, linestyle="-.", label="optimal source mean"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#ef4444", markersize=7, alpha=0.45, label="simulated choices"),
        Line2D([0], [0], color="#dc2626", lw=2.4, label=choice_mean_label),
    ]
    if is_probe:
        legend_items.append(Line2D([0], [0], color="#9ca3af", lw=2, linestyle="--", label="hidden source"))
    ax_circle.legend(handles=legend_items, loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)

    x = np.arange(N_SOURCES)
    bar_colors = [SOURCE_COLORS[source % len(SOURCE_COLORS)] for source in range(N_SOURCES)]
    weight_distributions = [
        weights_by_round[np.isfinite(weights_by_round[:, source]), source]
        for source in range(N_SOURCES)
    ]
    violin_parts = ax_weights.violinplot(
        weight_distributions,
        positions=x,
        widths=0.72,
        showmeans=False,
        showextrema=False,
        showmedians=False,
    )
    for source, body in enumerate(violin_parts["bodies"]):
        body.set_facecolor(bar_colors[source])
        body.set_edgecolor(bar_colors[source])
        body.set_alpha(0.34)
    ax_weights.boxplot(
        weight_distributions,
        positions=x,
        widths=0.18,
        showfliers=False,
        patch_artist=False,
        medianprops={"color": "#111827", "linewidth": 1.5},
        boxprops={"color": "#374151", "linewidth": 1.0},
        whiskerprops={"color": "#6b7280", "linewidth": 1.0},
        capprops={"color": "#6b7280", "linewidth": 1.0},
    )
    ax_weights.scatter(x, weights, color=bar_colors, edgecolor="#111827", linewidth=0.7, s=48, zorder=4)
    ax_weights.set_xticks(x, SOURCE_LABELS)
    ax_weights.set_ylim(0.0, 1.0)
    ax_weights.set_ylabel("weight")
    ax_weights.set_title(f"distribution of {weights_name.replace('_', ' ')} across trajectories")
    ax_weights.grid(axis="y", alpha=0.25)
    if is_probe:
        ax_weights.axvline(hidden_source, color="#111827", linewidth=1.2, linestyle="--", alpha=0.55)
    for source, weight in enumerate(weights):
        ax_weights.text(source, min(float(weight) + 0.03, 0.96), f"{weight:.2f}", ha="center", va="bottom", fontsize=9)

    fig.suptitle(
        "learningRate={learningRate:.2f}, g_y={g_y:.2f}, g_z={g_z:.2f}, "
        "vmConcentration={vmConcentration:.1f}, c0={c0:.2f}, trajectories={trajectories}".format(
            learningRate=learningRate,
            g_y=g_y,
            g_z=g_z,
            vmConcentration=vmConcentration,
            c0=c0,
            trajectories=N_ROUNDS,
        ),
        y=1.02,
    )
    matrix_image = draw_matrix_heatmap(
        ax_learned,
        learned_pair_state,
        "mean final c across trajectories",
        limit=matrix_limit,
    )
    draw_matrix_heatmap(
        ax_realized,
        stimulus_relative_agreement,
        "mean stimulus a across trials",
        limit=matrix_limit,
    )
    fig.colorbar(matrix_image, ax=[ax_learned, ax_realized], shrink=0.82, label="relative-agreement score")
    plt.show()


slider_style = {"description_width": "initial"}
slider_layout = {"width": "540px"}

learningRate_slider = FloatSlider(value=0.20, min=0.0, max=1.0, step=0.01, description="learningRate", continuous_update=False, readout_format=".2f", style=slider_style, layout=slider_layout)
g_y_slider = FloatSlider(value=1.0, min=0.0, max=100.0, step=0.05, description="g_y", continuous_update=False, readout_format=".2f", style=slider_style, layout=slider_layout)
g_z_slider = FloatSlider(value=1.0, min=0.0, max=100.0, step=0.05, description="g_z", continuous_update=False, readout_format=".2f", style=slider_style, layout=slider_layout)
vmConcentration_slider = FloatSlider(value=8.0, min=0.1, max=64.0, step=0.1, description="vmConcentration", continuous_update=False, readout_format=".1f", style=slider_style, layout=slider_layout)
c0_slider = FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.01, description="c0", continuous_update=False, readout_format=".2f", style=slider_style, layout=slider_layout)

participant_counter = BoundedIntText(
    value=0,
    min=0,
    max=N_PARTICIPANTS - 1,
    step=1,
    description="participant",
    style=slider_style,
    layout={"width": "180px"},
)

controls = HBox([
    VBox([learningRate_slider, g_y_slider, g_z_slider, vmConcentration_slider, c0_slider]),
    VBox([participant_counter]),
], layout={"align_items": "flex-start"})
out = interactive_output(
    draw_simulation,
    {
        "learningRate": learningRate_slider,
        "g_y": g_y_slider,
        "g_z": g_z_slider,
        "vmConcentration": vmConcentration_slider,
        "c0": c0_slider,
        "participant": participant_counter,
    },
)
display(VBox([controls, out]))